## Connect to WarehousePG
This Notebook is designed to be run on the Coordinator node. See GitHub repo for details on how to start the Notebook process.

In [ ]:
from sqlalchemy import create_engine
PGUSER="gpadmin"
PGHOST="cdw"
PGPORT="5432"
PGDATABASE="dev"
conn = create_engine(f"postgresql://{PGUSER}@{PGHOST}:{PGPORT}/{PGDATABASE}")

# If running elsewhere without trust auth, set a password and use this instead:
# PGPASSWORD="your_password_here"
# conn = create_engine(f"postgresql://{PGUSER}:{PGPASSWORD}@{PGHOST}:{PGPORT}/{PGDATABASE}")

%reload_ext sql
%sql conn

# `gpfdist`
Loading data with `gpfdist` protocol.

## Generate dataset of random data
Let's get started! First, we will create a 1 million row file with random data in it. Next, we will load that data into the database using an External Table with `gpfdist`. 

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)
N = 1_000_000

# ---- Reference data ----
categories = ['Electronics', 'Clothing', 'Home & Garden', 'Sports', 'Books', 'Toys', 'Beauty', 'Food & Beverage']
regions = ['North', 'South', 'East', 'West', 'Central']
payment_methods = ['Credit Card', 'Debit Card', 'PayPal', 'Cash', 'Bank Transfer']

category_price_range = {
    'Electronics': (50, 2000), 'Clothing': (10, 200), 'Home & Garden': (15, 500),
    'Sports': (10, 400), 'Books': (5, 60), 'Toys': (5, 150),
    'Beauty': (5, 100), 'Food & Beverage': (2, 50),
}

# Product catalog (300 products, each tied to a category/price range)
n_products = 300
product_catalog = pd.DataFrame({
    'product_id': np.arange(1, n_products + 1),
    'category': np.random.choice(categories, n_products),
})
product_catalog['base_price'] = product_catalog['category'].map(
    lambda c: np.random.uniform(*category_price_range[c])
)
product_catalog['product_name'] = product_catalog['category'] + '_Item_' + product_catalog['product_id'].astype(str)

# ---- Generate 1M orders ----
df = pd.DataFrame({
    'order_id': np.arange(1, N + 1),
    'product_id': np.random.choice(product_catalog['product_id'], size=N),
})
df = df.merge(product_catalog, on='product_id', how='left')

start_date = pd.Timestamp('2023-01-01')
end_date = pd.Timestamp('2025-12-31')
date_range_days = (end_date - start_date).days
df['order_date'] = start_date + pd.to_timedelta(np.random.randint(0, date_range_days, size=N), unit='D')

df['customer_id'] = np.random.randint(1, 200_001, size=N)
df['region'] = np.random.choice(regions, size=N)
df['sales_rep'] = np.random.choice([f'Rep_{i}' for i in range(1, 51)], size=N)
df['quantity'] = np.random.randint(1, 11, size=N)
df['unit_price'] = (df['base_price'] * np.random.uniform(0.9, 1.1, size=N)).round(2)
df['discount_pct'] = np.random.choice([0, 0, 0, 0.05, 0.1, 0.15, 0.2], size=N)
df['payment_method'] = np.random.choice(payment_methods, size=N)

df['gross_amount'] = (df['quantity'] * df['unit_price']).round(2)
df['discount_amount'] = (df['gross_amount'] * df['discount_pct']).round(2)
df['total_amount'] = (df['gross_amount'] - df['discount_amount']).round(2)

df = df.drop(columns=['base_price'])[
    ['order_id', 'order_date', 'customer_id', 'product_id', 'product_name', 'category',
     'region', 'sales_rep', 'quantity', 'unit_price', 'discount_pct',
     'gross_amount', 'discount_amount', 'total_amount', 'payment_method']
]

print(df.shape)
df.head()

## Export 1M rows
Python was used to generated the random data and it is stored in a DataFrame. Now, we will export the DataFrame to a CSV file.

In [ ]:
import csv
# output DataFrame to a file
df.to_csv("demo.csv", index=False, quoting=csv.QUOTE_NONNUMERIC)

## Create table to load data
Now, we will create a new schema called `loading_demo` and a table called `sales` to store the data.

In [ ]:
%%sql
DROP SCHEMA IF EXISTS loading_demo CASCADE;
CREATE SCHEMA loading_demo;

CREATE TABLE loading_demo.sales(   
    order_id int,
    order_date date,
    customer_id int,
    product_id int,
    product_name varchar,
    category varchar,
    region varchar,
    sales_rep varchar,
    quantity varchar,
    unit_price numeric,
    discount_pct numeric,
    gross_amount numeric,
    discount_amount numeric,
    total_amount numeric,
    payment_method varchar
)
DISTRIBUTED BY (order_id);


## Start `gpfdist`
gpfdist is a process that runs on an ETL server to serve files. This is a highly scalable process that allows segment processes to connect directly to fetch data. This removes the bottleneck of loading data through the coordinator.

We could use `COPY` to load the data but the preferred mechanism is to use `gpfdist`. 

For this demo, the gpfdist process will be run on the same node as the Notebook but it could easily be run on a dedicated node or set of nodes.

gpfdist is typically run as a background process from a shell like this:

`gpfdist -p 8899 -d /home/gpadmin/ > /home/gpadmin/gpfdist_8899.log 2>&1 &`

* -p indicates the port
* -d indicates the directory where files can be found by gpfdist
* stdout and stderr are being redirected to gpfdist_8999.log file

Note: The cell is using os.system to start gpfdist so that it can run as a background process.

In [ ]:
import os
os.system("""
pid=$(ps -ef | grep gpfdist | grep -v grep | awk -F ' ' '{print $2}')
if [ ! "$pid" == "" ]; then
    echo "kill $pid"
    kill "$pid"
    sleep 1
fi
echo "Starting gpfdist on port 8899 for files in /home/gpadmin/"
gpfdist -p 8899 -d /home/gpadmin/ > /home/gpadmin/gpfdist_8899.log 2>&1 < /dev/null &

pid=$(ps -ef | grep gpfdist | grep -v grep | awk -F ' ' '{print $2}')
echo "gpfdist pid: ${pid}"
""")

## Create External Table
Create an external table that uses the `gpfdist` process along with the `demo.csv` file.

Notice the `LOCATION` of the external table specifies the `gpfdist` protocol, the hostname, and the port number for the process just started.

The file being served is `demo.csv` which was created above.

In [ ]:
%%sql
CREATE EXTERNAL TABLE loading_demo.ext_sales (
    like loading_demo.sales)
LOCATION ('gpfdist://cdw:8899/demo.csv')
FORMAT 'CSV' (HEADER DELIMITER ',' NULL '' QUOTE '"');

## Test External Table
This queries the file directly. It uses the gpfdist process so that the segments can fetch data in parallel.

In [ ]:
%%sql
SELECT * FROM loading_demo.ext_sales LIMIT 10;

## Insert Data
Each segment connects in parallel to the gpfdist process to fetch the data. WarehousePG reads the data in parallel and then broadcasts it to every segment based on the hash of the distribution key.

In [ ]:
%%time
%%sql
INSERT INTO loading_demo.sales SELECT * FROM loading_demo.ext_sales;

## Validate data was loaded
The SQL below uses the WarehousePG table with the newly loaded data.

In [ ]:
%%sql
SELECT * FROM loading_demo.sales LIMIT 10;

## View data on each segment
Use the hidden column gp_segment_id, to see how many rows were loaded to each segment. In this example, there is even distribution across the segments.

This is also how WarehousePG scales to larger data sizes. Adding more nodes and more segments will spread the data across more segments.

In [ ]:
%%sql
SELECT gp_segment_id, count(*)
FROM loading_demo.sales
GROUP BY gp_segment_id
ORDER BY gp_segment_id;

## View Table Stats
Statistics are automatically gathered so that query plans are accurate and efficient. 

Note: statistics update asynchronously so it make take a minute to see the results.

In [ ]:
%%sql
SELECT reltuples AS rows
FROM pg_class c
JOIN pg_namespace n ON c.relnamespace = n.oid
WHERE n.nspname = 'loading_demo'
AND c.relname = 'sales';

In [ ]:
%%sql
SELECT *
FROM pg_stats 
WHERE schemaname = 'loading_demo'
AND tablename = 'sales';

## `gpfdist` cleanup
Now that we are done with loading data with `gpfdist`, let's stop that process.

In [ ]:
%%bash
pid=$(ps -ef | grep gpfdist | grep -v grep | awk -F ' ' '{print $2}')
if [ ! "$pid" == "" ]; then
    echo "kill $pid"
    kill "$pid"
fi

# Platform Extension Framework (PXF) 
Loading data with the PXF protocol.

Note: This part of the Notebook assumes you already have PXF installed.

## Configure PXF
PXF is a very flexible extension that can be used to load data from various sources like JDBC and S3. The following is an example of using PXF to load from the AWS Open Data Registry.

PXF configuration files are stored as "servers" in the `$PXF_BASE` directory. We will create a new server that connects to a bucket in us-west-2. For the demo, we are using a public bucket that is part of the AWS Open Data Registry.

Source: https://registry.opendata.aws/prod-comp-shopping/

In [ ]:
%%bash

if [ "${PXF_BASE}" == "" ]; then
    PXF_BASE=$(cd $(dirname $(readlink -f $(command -v pxf)))/.. && pwd)
fi

SERVER_NAME="pxf_s3demo_uswest2"

mkdir -p ${PXF_BASE}/servers/${SERVER_NAME}

cat > ${PXF_BASE}/servers/${SERVER_NAME}/s3-site.xml << 'EOF'
<?xml version="1.0"?>
<configuration>
  <property>
    <name>fs.s3a.endpoint</name>
    <value>s3.us-west-2.amazonaws.com</value>
  </property>
  <property>
    <name>fs.s3a.aws.credentials.provider</name>
    <value>org.apache.hadoop.fs.s3a.AnonymousAWSCredentialsProvider</value>
  </property>
</configuration>
EOF

pxf cluster sync
pxf cluster restart

## PXF External Table
The External Table here is similar to the one that uses `gpfdist` protocol but with this example, the `LOCATION` uses the `pxf` protocol.
* `prod-comp-shopping-dataset` is the S3 bucket 
* `final_prodcomp_dataset_cleaned.tsv` is the file in the S3 bucket
* `PROFILE=s3:text` indicates PXF should use the S3 text profile
* `SERVER=pxf_s3demo_uswest2` is using the newly added server that uses anonymous AWS credentials
* `SKIP_HEADER_COUNT=1` indicates to skip the first row because it is a header row

In [ ]:
%%sql
DROP EXTERNAL TABLE IF EXISTS loading_demo.ext_pxf_example;

CREATE EXTERNAL TABLE loading_demo.ext_pxf_example (
    attribute_names_and_values_for_products_under_consideration varchar,
    template_comparative_sentence varchar,
    ground_truth_comparitive_sentence varchar)
LOCATION ('pxf://prod-comp-shopping-dataset/final_prodcomp_dataset_cleaned.tsv?PROFILE=s3:text&SERVER=pxf_s3demo_uswest2&SKIP_HEADER_COUNT=1')
FORMAT 'TEXT' (delimiter E'\t' escape 'OFF');

## Test External Table from S3 using PXF
This query shows how PXF is used by the Segments to connect to S3 in parallel to fetch the data.

In [ ]:
%%sql
SELECT *
FROM loading_demo.ext_pxf_example
LIMIT 10;

## `jsonb` Datatype
First, let's create the WarehousePG table that will store the data. Notice how the first column uses `jsonb` datatype while the External Table above uses the `varchar` datatype. More on this below!

In [ ]:
%%sql
DROP TABLE IF EXISTS loading_demo.pxf_example;

CREATE TABLE loading_demo.pxf_example (
    attribute_names_and_values_for_products_under_consideration jsonb,
    template_comparative_sentence varchar,
    ground_truth_comparitive_sentence varchar)
DISTRIBUTED RANDOMLY;

## User Defined Function
The free sample data has JSON data but it isn't formed properly. It has single quotes instead of double so it won't directly load into a column defined as `jsonb`. 
To fix this, a user defined function written in Python can properly format the data. It executes in WarehousePG and on each segment in the cluster. The parallel execution of the user defined function inside the cluster is key to the performance of this transformation.

WarehousePG is an ideal tool for ELT. This is where you transform the data using the parallel architecture.

In [ ]:
%%sql

CREATE OR REPLACE FUNCTION loading_demo.pydict_to_jsonb(s text)
RETURNS jsonb
LANGUAGE plpython3u
AS $$
import ast, json
if s is None:
    return None
return json.dumps(ast.literal_eval(s))
$$;

## Load data from the S3 bucket
Notice how the data is transformed as it is inserted with utilizing SQL. Also notice how fast the loading with transformation executed!

In [ ]:
%%time
%%sql

--truncate is only here to allow this cell to be executed repeatedly.
TRUNCATE loading_demo.pxf_example;

INSERT INTO loading_demo.pxf_example (
    attribute_names_and_values_for_products_under_consideration,
    template_comparative_sentence,
    ground_truth_comparitive_sentence)
SELECT loading_demo.pydict_to_jsonb(attribute_names_and_values_for_products_under_consideration) as attribute_names_and_values_for_products_under_consideration,
    template_comparative_sentence,
    ground_truth_comparitive_sentence
FROM loading_demo.ext_pxf_example;

## Show a sample of the data

In [ ]:
%%sql
SELECT *
FROM loading_demo.pxf_example
LIMIT 10;

## Parse and filter the JSON column 
This is standard PostgreSQL syntax for handling JSON data so if you are familiar with PostgreSQL, WarehousePG makes the analytics very easy because the SQL syntax will be the same.

In [ ]:
%%sql
SELECT template_comparative_sentence, ground_truth_comparitive_sentence, 
    attribute_names_and_values_for_products_under_consideration ->> 'absorbency rating_product1' AS absorbency_rating_product1 
FROM loading_demo.pxf_example
WHERE attribute_names_and_values_for_products_under_consideration ? 'absorbency rating_product1';

## Close the connection

In [ ]:
connection_url = f"postgresql://{PGUSER}@{PGHOST}:{PGPORT}/{PGDATABASE}"
%sql --close {{connection_url}}

# Finished!
You did it! You successfully loaded data using the two most common patterns. The first uses `gpfdist` to make files available in External Tables and the second uses PXF to make files available from S3 in External Tables.

Both protocols support additional capabilities including reading compressed files, JDBC connections, and more! 